# Full HelpSteer2 All-Method Merge Evaluation

This notebook runs the full HelpSteer2 all-method adapter-merge evaluation for the thesis prototype. It evaluates coefficient choices inside the fixed Rewarded-Soups-style interpolation family by merging the five trained HelpSteer2 LoRA adapters, generating answers on the fixed prompt set, computing proxy scores, and summarizing utility.

The scores are proxy scores, not HelpSteer2 human labels and not reward-model scores. The results should not be described as global Pareto-front improvement. Do not commit `adapters/`, zip files, safetensors, bin files, checkpoints, or model weights.


## 1. Clone or update the repository

The repository is placed at `/content/master-thesis`. The cell avoids nested folders such as `/content/master-thesis/master-thesis`.


In [ ]:
from pathlib import Path
import os
import subprocess

repo_path = Path("/content/master-thesis")
repo_url = "https://github.com/NZhang137/master-thesis.git"

os.chdir("/content")

if (repo_path / ".git").is_dir():
    print("Repository found. Pulling latest changes...")
    subprocess.run(["git", "-C", str(repo_path), "pull"], check=True)
elif repo_path.exists():
    raise RuntimeError(
        f"{repo_path} exists but is not a Git repository. "
        "Rename or remove it, then run this cell again."
    )
else:
    print("Cloning repository...")
    subprocess.run(["git", "clone", repo_url, str(repo_path)], check=True)

os.chdir(repo_path)
print(f"Current folder: {Path.cwd()}")


## 2. Check GPU

A GPU is strongly recommended for this full evaluation run.


In [ ]:
!nvidia-smi


## 3. Install dependencies

These packages are needed for GPT-2, PEFT adapter loading, generation, pandas previews, and evaluation. Pandas and NumPy are pinned to Colab-friendly versions. If you later see a `torchao` version error, run `!pip install -q -U torchao` and restart the runtime.


In [ ]:
!pip install -q "pandas==2.2.2" "numpy<2.1" transformers datasets peft accelerate safetensors scipy


## 4. Show important files

Check that the scripts, result inputs, and fixed prompt file are visible.


In [ ]:
!pwd
!ls
!ls scripts
!ls results
!ls data/evaluation_prompts


## 5. Upload and unzip the adapter backup

Upload your local `helpsteer2_adapters.zip` manually in the notebook when prompted. The notebook also accepts `helpsteer2_adapters_longrun.zip` if that file exists. The zip is only a temporary Colab backup and should not be committed.

The upload cell moves the zip to `/content/`, outside the repository. The evaluation script expects the five adapter folders under `/content/master-thesis/adapters/`. The normalization cell below searches for the adapter folders after unzipping and copies them to the expected location if the zip created a nested folder.


In [ ]:
from pathlib import Path
from google.colab import files
import shutil

zip_candidates = [
    Path("/content/helpsteer2_adapters.zip"),
    Path("/content/helpsteer2_adapters_longrun.zip"),
]

if any(path.is_file() for path in zip_candidates):
    print("Adapter zip already exists in /content/.")
else:
    print("Upload helpsteer2_adapters.zip now.")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No file was uploaded.")

    accepted_names = {"helpsteer2_adapters.zip", "helpsteer2_adapters_longrun.zip"}
    moved_any = False
    for uploaded_name in uploaded.keys():
        uploaded_path = Path(uploaded_name)
        if uploaded_path.name not in accepted_names:
            print(f"Ignoring unexpected uploaded file: {uploaded_path.name}")
            continue

        target_path = Path("/content") / uploaded_path.name
        if uploaded_path.resolve() != target_path.resolve():
            shutil.move(str(uploaded_path), str(target_path))
        moved_any = True
        print(f"Saved temporary adapter zip at {target_path}")

    if not moved_any:
        raise FileNotFoundError("Please upload helpsteer2_adapters.zip.")


In [ ]:
from pathlib import Path

zip_candidates = [
    Path("/content/helpsteer2_adapters.zip"),
    Path("/content/helpsteer2_adapters_longrun.zip"),
]
adapter_zip = next((path for path in zip_candidates if path.is_file()), None)

if adapter_zip is None:
    raise FileNotFoundError(
        "Please upload helpsteer2_adapters.zip to /content/ first. "
        "The notebook also accepts helpsteer2_adapters_longrun.zip."
    )

print(f"Using adapter zip: {adapter_zip}")
!ls -lh {adapter_zip}
!unzip -l {adapter_zip} | head -n 40
!unzip -o {adapter_zip} -d /content/master-thesis


In [ ]:
from pathlib import Path
import shutil

repo_path = Path("/content/master-thesis")
target_root = repo_path / "adapters"
attributes = ["helpfulness", "correctness", "coherence", "complexity", "verbosity"]
expected_names = [f"helpsteer2-gpt2-{attribute}-adapter" for attribute in attributes]

def has_all_adapters(root):
    return all((root / name).is_dir() for name in expected_names)

if has_all_adapters(target_root):
    print(f"Adapters already found at {target_root}")
else:
    candidate_roots = []
    for helpfulness_dir in Path("/content").rglob("helpsteer2-gpt2-helpfulness-adapter"):
        root = helpfulness_dir.parent
        if has_all_adapters(root):
            candidate_roots.append(root)

    if not candidate_roots:
        raise FileNotFoundError(
            "Could not find all five HelpSteer2 adapter folders under /content. "
            "Check that you uploaded the correct helpsteer2_adapters.zip file."
        )

    source_root = candidate_roots[0]
    print(f"Found adapter folders at {source_root}")
    target_root.mkdir(parents=True, exist_ok=True)

    if source_root.resolve() != target_root.resolve():
        for name in expected_names:
            source = source_root / name
            target = target_root / name
            if target.exists():
                shutil.rmtree(target)
            shutil.copytree(source, target)
        print(f"Copied adapter folders to {target_root}")

print("Expected adapter folders:")
for name in expected_names:
    print(" -", target_root / name, "OK" if (target_root / name).is_dir() else "MISSING")


## 6. Verify adapters

This checks that all five HelpSteer2 PEFT adapter folders contain the expected adapter files.


In [ ]:
!python scripts/check_helpsteer2_adapters.py


## 7. Verify required input files

The full evaluation needs the coefficient table, method-cost table, and fixed prompt JSONL file.


In [ ]:
from pathlib import Path

required_files = [
    Path("results/helpsteer2_all_method_coefficients.csv"),
    Path("results/helpsteer2_method_costs.csv"),
    Path("data/evaluation_prompts/helpsteer2_fixed_prompts.jsonl"),
]

for file_path in required_files:
    if not file_path.is_file():
        raise FileNotFoundError(f"Missing required file: {file_path}")
    print(f"Found: {file_path}")


## 8. Run the full evaluation

This evaluates every coefficient row on every fixed prompt. It may take a while because the script repeatedly creates weighted adapter merges and generates answers.


In [ ]:
!python scripts/evaluate_helpsteer2_all_method_merges.py


## 9. Display output files


In [ ]:
!ls results

from pathlib import Path

expected_outputs = [
    Path("results/helpsteer2_all_method_generations.csv"),
    Path("results/helpsteer2_all_method_scores.csv"),
    Path("results/helpsteer2_all_method_result_summary.csv"),
    Path("results/helpsteer2_all_method_result_summary.md"),
    Path("results/helpsteer2_all_method_result_summary.json"),
]
missing_outputs = [path for path in expected_outputs if not path.is_file()]
if missing_outputs:
    raise FileNotFoundError(
        "The full evaluation did not create all expected output files. "
        "Go back to section 8 and fix the first error shown there. Missing: "
        + ", ".join(str(path) for path in missing_outputs)
    )

!head results/helpsteer2_all_method_generations.csv
!head results/helpsteer2_all_method_scores.csv
!cat results/helpsteer2_all_method_result_summary.md


## 10. Load and inspect the result summary

The summary contains mean proxy utility, distances to the original preference vector, and joined cost fields when available.


In [ ]:
import pandas as pd

summary = pd.read_csv("results/helpsteer2_all_method_result_summary.csv")
display(summary)


## 11. Best method per preference


In [ ]:
best_by_preference = (
    summary.sort_values("mean_utility", ascending=False)
    .groupby("preference_name", as_index=False)
    .first()
)

display(best_by_preference[[
    "preference_name",
    "method",
    "hyperparameter_id",
    "mean_utility",
    "l1_distance_to_p",
    "l2_distance_to_p",
]])


## 12. Mean utility by method


In [ ]:
utility_by_method = (
    summary.groupby("method", as_index=False)["mean_utility"]
    .mean()
    .sort_values("mean_utility", ascending=False)
)
display(utility_by_method)


## 13. Improvements over direct preference and uniform


In [ ]:
improvement_columns = [
    column
    for column in [
        "preference_name",
        "method",
        "hyperparameter_id",
        "mean_utility",
        "improvement_over_direct_preference",
        "improvement_over_uniform",
    ]
    if column in summary.columns
]

display(summary[improvement_columns].sort_values(
    ["preference_name", "mean_utility"],
    ascending=[True, False],
))


## 14. Runtime and cost columns


In [ ]:
cost_columns = [
    column
    for column in [
        "preference_name",
        "method",
        "hyperparameter_id",
        "runtime_seconds",
        "peak_memory_mb",
        "solver_iterations",
        "solver_success",
    ]
    if column in summary.columns
]

display(summary[cost_columns].head(30))

if "runtime_seconds" in summary.columns:
    display(summary.groupby("method", as_index=False)["runtime_seconds"].mean())


## 15. Git safety check

Small CSV, JSON, and Markdown result files may be committed if useful. Do not commit `adapters/`, zip files, safetensors, bin files, checkpoints, or model weights.


In [ ]:
!git status


## Next step

After the full evaluation finishes, analyze proxy scores, preference-weighted utility, distance to `p`, prompt-category behavior, and runtime/cost columns. Then update the HelpSteer2 result report with careful wording about proxy-score limitations and the fixed interpolation-family setting.
